In [1]:
"""
This script creates artificial data for a discrete choice problem.
Assume there are three modes of transportation to choose from. Six fixed
variables were designed as significant and five as non-significant.
Seven random variables (five normal, one uniform, one triangular) were
designed as significant.
Three normal variables were correlated.
Two normal variables were non-linearly transformed.
"""

from sklearn import datasets





In [2]:

import numpy as np

import random


random_state = None


import statsmodels.api as sm
from scipy.stats import multivariate_normal

In [7]:
import numpy as np
import pandas as pd
import scipy.stats as ss

from sklearn import preprocessing
 

is_correlated = True
if is_correlated:
    name_2 = 'artificial_mixed_corr_2023_MOOF.csv'
else:
    name_2 = 'artificial_ZA.csv'


In [3]:
def noise(n_obs, perc=1, random_state=None):
    random_state = np.random
    noise_vec = random_state.normal(0, perc, n_obs)
    return noise_vec

In [9]:
def random_col(N, P, J, random_state=np.random, noise_val = 0.25):
    rand_nums = random_state.uniform(low=0, high=1, size=(P,))/100
    rand_nums = [random.randint(0, 1) for _ in range(N)]
    return np.tile(rand_nums, P) + noise_val*noise(N*P*J, random_state=random_state)

def generate_random_df(N, P, J, num_fixed=0, num_isvars=0, num_randvars=0, random_state=None, add_constant = 1, non_sig_num = None):
    df = pd.DataFrame()
    df['id'] = np.repeat(np.arange(1, (N*P+1)), J)
    #df['ind_id'] = np.repeat(np.arange(1, N+1), P*J)
    #df['alt'] = np.tile(np.arange(1, J+1), N*P)
    X, y = datasets.make_regression(n_samples=N, n_features=6,n_informative=6, noise=0.0)
    varnames = []
    col = random_col(num_fixed, P, J, random_state, 0)
    col = [random.random() for _ in range(num_fixed+num_randvars)]
    col = list(range(1, num_fixed+num_randvars+1))
 
# Shuffle the list
    random.shuffle(col)
  
    #X = (np.random.multivariate_normal(mean=col, cov=np.eye(len(col)), size=N))
   
    #X = scaler.fit_transform(X)
    print(X.shape)
    for i in range(num_fixed):
      
        coef_name = 'fixed' + str(i+1)
        varnames.append(coef_name)
       # random_col(N, P, J, random_state=random_state, 0)
        df[coef_name] = random_col(N, P, J, random_state=np.random, noise_val = 0)
       
        df[coef_name] = X[:,i]
        

    for i in range(num_isvars):
        coef_name = 'added_isvar' + str(i+1)
        varnames.append(coef_name)
        col_vals = np.repeat(random_state.random(N*P)*100, J)
        for j in range(J):
            if j == 0:
                df[coef_name] = col_vals
            else:
                df[coef_name + "." + str(j+1)] = col_vals

    #ol = random_col(num_randvars, P, J, random_state)
    #col = [random.random() for _ in range(num_randvars)]
    #X = (np.random.multivariate_normal(mean=col, cov=np.eye(num_randvars), size=N)+1)/2
    
    for i in range(num_randvars):
        coef_name = 'random' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=np.random, noise_val = 0)
        df[coef_name] =X[:,i]+num_fixed
        

    if add_constant:
        df = sm.add_constant(df, prepend=False)
    
    for i in range(non_sig_num):
        coef_name = 'non_sig' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state) 
    print(varnames)
   
    for tr in [varnames]:
        scaler = preprocessing.MinMaxScaler()
        scaler.fit(df[tr])
        df[tr] = scaler.transform(df[tr])
    
       
        
    return df, varnames


In [10]:

N =1000  # Number of observations
P = 1  # Number of choices per individual
J = 1  # Number of alternatives
num_fixed = 3
num_isvars = 0
num_nonsig = 5
num_randvars = 3
seed = 122
random_state = np.random.RandomState(seed)
np.random.seed(seed)

random.seed(seed)
df, varnames = generate_random_df(N, P, J, num_fixed=num_fixed, num_isvars=num_isvars,
                                  num_randvars=num_randvars, random_state=random_state, non_sig_num = num_nonsig)

# Shift values <-2 for as inverse boxcox transform will be applied





(1000, 6)
['fixed1', 'fixed2', 'fixed3', 'random1', 'random2', 'random3', 'non_sig1', 'non_sig2', 'non_sig3', 'non_sig4', 'non_sig5']


In [158]:
df.head(100)

,id,fixed1,fixed2,fixed3,random1,random2,random3,const,non_sig1,non_sig2,non_sig3,non_sig4,non_sig5
0,1,0.715971,0.433123,0.426991,0.715971,0.433123,0.426991,1.0,0.306963,0.667900,0.699710,0.249670,0.659188
1,2,0.473305,0.322747,0.529922,0.473305,0.322747,0.529922,1.0,0.320759,0.215301,0.298543,0.075275,0.522105
2,3,0.479413,0.437658,0.437795,0.479413,0.437658,0.437795,1.0,0.707647,0.301741,0.686057,0.764354,0.885599
3,4,0.660747,0.571491,0.310612,0.660747,0.571491,0.310612,1.0,0.302694,0.308708,0.764751,0.424800,0.624104
4,5,0.086128,0.622603,0.425840,0.086128,0.622603,0.425840,1.0,0.592822,0.546501,0.509390,0.327721,0.159603
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0.702822,0.652243,0.322877,0.702822,0.652243,0.322877,1.0,0.267480,0.772064,0.753310,0.569463,0.687969
96,97,0.447283,0.331982,0.380906,0.447283,0.331982,0.380906,1.0,0.143863,0.223256,0.836596,0.504347,0.708141
97,98,0.518605,0.507775,0.502420,0.518605,0.507775,0.502420,1.0,0.618990,0.588647,0.217254,0.354056,0.691940
98,99,0.283865,0.332898,0.578352,0.283865,0.332898,0.578352,1.0,0.196743,0.213290,0.426375,0.280432,0.642434


In [11]:
# Define coefficients (betas)
# Fixed betas
fixed_coefs = [ random_state.choice([-1,1, -1]) *random_state.uniform(1, 4) for i in range(num_fixed)]

fixed_coefs = [0, 0, 0]
fixed_coefs = np.array(fixed_coefs)



fixed_coefs = list(fixed_coefs)


In [12]:
print(fixed_coefs)


print(df.head())

[0, 0, 0]
   id    fixed1    fixed2    fixed3   random1   random2   random3  const  \
0   1  0.715971  0.433123  0.426991  0.715971  0.433123  0.426991    1.0   
1   2  0.473305  0.322747  0.529922  0.473305  0.322747  0.529922    1.0   
2   3  0.479413  0.437658  0.437795  0.479413  0.437658  0.437795    1.0   
3   4  0.660747  0.571491  0.310612  0.660747  0.571491  0.310612    1.0   
4   5  0.086128  0.622603  0.425840  0.086128  0.622603  0.425840    1.0   

   non_sig1  non_sig2  non_sig3  non_sig4  non_sig5  
0  0.306963  0.667900  0.699710  0.249670  0.659188  
1  0.320759  0.215301  0.298543  0.075275  0.522105  
2  0.707647  0.301741  0.686057  0.764354  0.885599  
3  0.302694  0.308708  0.764751  0.424800  0.624104  
4  0.592822  0.546501  0.509390  0.327721  0.159603  


In [20]:
# Random mean between -1.5 and 1.5, excluding -.1 - .1 as hard to detect effect
random_coefs_mean = [random_state.choice([-1,1,-1,1, 1, 1]) * random_state.uniform(1, 5) for i in range(num_randvars)]
random_coefs_mean[1] = -5.5
random_coefs_mean[2] = 2
random_coefs_mean = [1.22, -2.79, 2.67]
random_coefs_mean2 = [0, 0, 0]

#print(random_coefs_mean)
print(is_correlated)
cov_mat = np.diag(random_coefs_mean2).astype(float)
if is_correlated:

    #cov_mat = np.diag(random_coefs_sd)
    cov_mat[0, 0] = 2
    cov_mat[1,1]  =  1
    cov_mat[2, 2] = 4
    
    
    cov_mat[0, 1]  = .2
    cov_mat[0, 1] = cov_mat[1, 0] = .2
    cov_mat[0, 2] = cov_mat[2, 0] = .3
    cov_mat[1, 2] = cov_mat[2, 1] = .4



    print(cov_mat)


rand_coefs = [np.array([]) for i in range(num_randvars)]

X = df.values[:, 1:num_fixed+num_randvars+2]  # Extract only necessary columns


True
[[2.  0.2 0.3]
 [0.2 1.  0.4]
 [0.3 0.4 4. ]]


In [162]:
Monte = 1000
yMonte = []

for i in range(Monte):
    rand_coefs = [np.array([]) for i in range(num_randvars)]
    for i in range(N):
        
        res_normal = random_state.multivariate_normal(random_coefs_mean, cov_mat)
        #print(res_normal)
        #res_normal = random_coefs_mean + res_normal
        
        
        res = res_normal
        for r in range(num_randvars):
            rand_coefs[r] = np.append(rand_coefs[r], np.repeat(res[r], P*J))

    B_fixed = [np.repeat(f_coef, P*N*J) for f_coef in fixed_coefs]
    B_const = [np.repeat(-2.29, P*N*J)]
    if is_correlated:
        B_const = [np.repeat(-3, P*N*J)]
    else:
        B_const = [np.repeat(-2, P*N*J)]
        

    # Convert betas to matrix for easy product
    B = [B_fixed, rand_coefs, B_const]
   # B = [B_i for B_i in B if B_i != []]
 
    B = np.vstack(B).T
    

    # Multiply and generate probability
    #isvars = ['added_isvar' + str(i+1) for i in range(num_isvars)]

    dispersion = 2
    XB = (X*B).sum(axis=1).reshape(N*P, J)
    nb = 1
    if nb:
        eps = np.random.gumbel(0, 1, (N*P, J))
        eps = np.log(np.random.gamma(1, dispersion, N))
        #print(eps)
        eXB = np.exp(XB+eps.reshape(N*P, J)).ravel()
    else:    
        eXB = np.exp(XB).ravel()
   

    
    #xg = np.array(rgamma(N, dispersion, dispersion))
    xbg = eXB
    #xbg = eXB

    
    nbyo = np.random.poisson(xbg)
    #nbyo = xbg
    yMonte.append(nbyo)


nbyo = np.mean(yMonte, axis = 0)

In [163]:


# Use monte carlo simulation to predict choice
# y = np.apply_along_axis(lambda p: np.eye(J, dtype='int64')[np.argmax(p)], 1, prob).reshape(N*P*J,)
# y = y.reshape(N*P*J,)


df['Y'] = nbyo
print(nbyo)

# Apply non-linear transformations for boxcox testing
save = "C:/Users/n9471103/source/repos/MetaCount/" +name_2
# Save to CSV
print(df.head())
df = df.drop(columns = ['id'])
print(df.head())
df.to_csv(save, index=False)

[ 0.563  0.621  0.357  0.26   0.094  0.159  0.067  0.4    0.503  1.075
  0.6    0.301  0.367  0.79   0.79   0.609  0.167  1.155  0.558  0.519
  1.307  0.638  0.286  0.543  1.038  0.261  0.541  0.079  0.779  0.706
  1.038  0.35   0.21   2.564  0.198  0.359  0.255  0.068  1.842  2.005
  0.393  3.105  0.068  0.344  1.125  0.246  0.576  0.163  0.698  0.319
  1.988  0.274  0.357  0.676  0.193  1.077  0.216  0.542  0.432  0.324
  0.695  0.182  1.025  0.592  0.485  1.865  0.203  1.291  0.271  0.425
  0.286  0.553  0.316  0.774  0.516  0.451  0.361  0.559  0.24   0.309
  0.276  0.161  0.364  0.48   0.257  1.262  0.247  2.182  0.291  4.133
  0.321  0.54   0.312  1.402  0.408  0.192  0.353  0.414  0.542  0.483
  0.609  1.004  0.58   0.987  0.184  1.158  1.627  2.538  0.25   2.31
  0.46   0.434  0.825  4.674  0.385  0.088  0.179  2.403  0.321  0.593
  0.547  1.03   0.525  0.097  0.45   0.856  0.252  1.823  0.624  0.673
  0.676  0.481  1.21   0.201  0.181  0.23   1.879  0.652  0.42   1.141
  0.421

In [164]:
np.shape(yMonte)

(1000, 1000)

In [165]:
np.array([0])

array([0])

In [166]:
a = np.mean(yMonte, axis=0)
sum(a)

762.7739999999982

In [167]:
# np.linalg.norm(model.coeff_[:11] - fixed_coefs)